# DeepSeek-V4-Flash를 SageMaker 실시간 엔드포인트로 배포하기

이 노트북은 로컬(RTX 6000 Pro ×2)에서 먼저 검증한 vLLM 서빙 구성을 SageMaker 실시간 엔드포인트로 옮기는 과정입니다.

> ⚠️ **과금 경고** — SageMaker 실시간 엔드포인트는 **삭제하기 전까지 시간당 계속 과금**됩니다. 다 쓰면 반드시 이 노트북 맨 아래 [Cleanup](#cleanup) 셀을 실행하세요. `p5en.48xlarge`(8×H200) 기준 시간당 비용이 상당히 높습니다 — 테스트만 할 거라면 인스턴스 크기를 줄이는 것도 고려하세요.

## 로컬 검증에서 이미 확인한 것들

이 노트북의 설정값은 즉흥적으로 고른 게 아니라, 로컬 vLLM 서빙([`../README.md`](../README.md))에서 실제로 부딪혀서 확인한 결론을 그대로 반영합니다.

| 결론 | 왜 |
|---|---|
| vLLM 이미지는 최신(`0.26.0`)이 아니라 `0.25.x` 계열 | `0.26.0`은 DeepSeek-V4에 FlashMLA 회귀 버그가 있음 |
| 스펙 디코딩은 **MTP**로 설정 (DSpark 아님) | `DeepSeek-V4-Flash-0731`은 DSpark 모듈만 내장하는데, DSpark는 SM120(RTX 6000 Pro 등) 계열 GPU에서 FlashInfer 커널 누락으로 크래시함 — [`../troubleshooting/dspark-sm120-crash.md`](../troubleshooting/dspark-sm120-crash.md). **H200/H100(SM90)에서는 이 버그가 없어 MTP든 DSpark든 정상 동작**하지만, 이 노트북은 로컬 검증과 동일한 설정을 유지하기 위해 MTP를 그대로 씁니다 |
| `kv-cache-dtype`은 `fp8`로 명시 | `auto`로 두면 일부 환경에서 자동 해석이 실패함 |

## 진행 순서

1. 설치 및 SageMaker 세션 준비
2. 인스턴스·모델 변수 설정
3. vLLM 서빙 환경변수 구성 (성능 프리셋 3가지 중 택1)
4. 모델 생성 → 엔드포인트 배포
5. 추론 테스트 (기본 / reasoning / 스트리밍)
6. **Cleanup** (필수)

## 1. 설치 및 SageMaker 세션 준비

In [ ]:
%pip install -q boto3

In [ ]:
import json
import time
import boto3

boto_session = boto3.Session()
region = boto_session.region_name
sm = boto_session.client("sagemaker")
sm_runtime = boto_session.client("sagemaker-runtime")

print(f"region: {region}")

In [ ]:
def get_sagemaker_role():
    """노트북 인스턴스/스튜디오에 붙은 role이 없으면 STS로 현재 identity를 사용."""
    try:
        import sagemaker
        return sagemaker.get_execution_role()
    except Exception:
        sts = boto_session.client("sts")
        return sts.get_caller_identity()["Arn"]


def _wait_for_resource(describe_fn, key, name, target_states, failure_states, poll_sec=30):
    while True:
        resp = describe_fn(**{name: key})
        status = resp["EndpointStatus"] if "EndpointStatus" in resp else resp["StatusMessage"]
        print(f"  status: {status}")
        if status in target_states:
            return resp
        if status in failure_states:
            raise RuntimeError(f"{key} entered failure state: {status}\n{resp.get('FailureReason', '')}")
        time.sleep(poll_sec)


def wait_for_endpoint(endpoint_name):
    print(f"Waiting for endpoint '{endpoint_name}' ...")
    return _wait_for_resource(
        lambda EndpointName: sm.describe_endpoint(EndpointName=EndpointName),
        endpoint_name, "EndpointName",
        target_states=["InService"], failure_states=["Failed"],
    )


role = get_sagemaker_role()
print(f"role: {role}")

## 2. 인스턴스·모델 변수 설정

DeepSeek-V4-Flash 가중치는 148.7 GiB입니다. `serving.md` 사이징 계산에 따르면 **G7e(RTX 6000 Pro급) 인스턴스로도 충분**하지만, 이 노트북은 로컬 검증(RTX 6000 Pro ×2, 96 GiB×2)보다 여유 있는 구성으로 SageMaker에 배포하는 예시로 `ml.p5en.48xlarge`(8×H200, 1128 GiB)를 기본값으로 둡니다.

> 💡 **더 저렴하게 배포하고 싶다면** — `ml.g6e.12xlarge`(2×L40S) 같은 더 작은 인스턴스로도 TP=2 구성이 가능합니다. 다만 SageMaker의 `ml.g7e` 계열 지원 여부는 EC2와 별개로 확인이 필요합니다([`../serving.md`](../serving.md) 참고).

In [ ]:
instance = {"type": "ml.p5en.48xlarge", "num_gpu": 8}

model_id = "deepseek-ai/DeepSeek-V4-Flash-0731"

timestamp = time.strftime("%Y%m%d-%H%M%S")
model_name = f"deepseek-v4-flash-{timestamp}"
endpoint_config_name = f"deepseek-v4-flash-config-{timestamp}"
endpoint_name = f"deepseek-v4-flash-{timestamp}"
variant_name = "AllTraffic"

hf_token = "<YOUR_HF_TOKEN_HERE>"  # DeepSeek-V4-Flash-0731은 gated가 아니지만, rate limit 완화를 위해 넣는 것을 권장

## 3. vLLM 서빙 환경변수 구성

vLLM DLC(Deep Learning Container)는 `SM_VLLM_*` 환경변수를 `vllm serve`의 CLI 플래그로 그대로 변환합니다. 로컬에서 `vllm serve`에 직접 넘긴 플래그와 1:1로 대응됩니다.

**성능 프리셋 3가지 중 하나를 고르세요** (기본은 latency 최적화):

| 프리셋 | 언제 쓰나 | 방법 |
|---|---|---|
| **Latency** (기본) | 응답 하나가 최대한 빨리 와야 할 때 | `SM_VLLM_TENSOR_PARALLEL_SIZE`만 설정 |
| **Balanced** | 지연시간과 처리량을 절충 | TP + `SM_VLLM_ENABLE_EXPERT_PARALLEL=true` |
| **Throughput** | 동시 요청이 많고 총 처리량이 중요할 때 | TP 제거, EP + `SM_VLLM_DATA_PARALLEL_SIZE` 설정 |

아래 셀에서 `PERFORMANCE_PRESET` 값만 바꾸면 됩니다.

In [ ]:
PERFORMANCE_PRESET = "latency"  # "latency" | "balanced" | "throughput"

inference_image = (
    f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.25.0-gpu-py312-cu130-ubuntu22.04-sagemaker"
)

# MTP 스펙 디코딩: DeepSeek-V4-Flash-0731은 DSpark 모듈도 내장하지만,
# SM120(RTX 6000 Pro 등)에서 DSpark가 크래시하는 것으로 확인됨 (../troubleshooting/dspark-sm120-crash.md).
# H200/H100(SM90)에서는 DSpark도 정상 동작하나, 로컬 검증과 동일하게 MTP를 사용.
spec_config = {"method": "mtp", "num_speculative_tokens": 3}

common_env = {
    "HF_TOKEN": hf_token,
    "SM_NUM_GPUS": json.dumps(instance["num_gpu"]),
}

vllm_env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_MAX_MODEL_LEN": "131072",
    "SM_VLLM_TRUST_REMOTE_CODE": "true",
    "SM_VLLM_KV_CACHE_DTYPE": "fp8",
    "SM_VLLM_BLOCK_SIZE": "256",
    "SM_VLLM_TOKENIZER_MODE": "deepseek_v4",
    "SM_VLLM_TOOL_CALL_PARSER": "deepseek_v4",
    "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
    "SM_VLLM_REASONING_PARSER": "deepseek_v4",
    "SM_VLLM_SPECULATIVE_CONFIG": json.dumps(spec_config),
}

if PERFORMANCE_PRESET == "latency":
    vllm_env["SM_VLLM_TENSOR_PARALLEL_SIZE"] = json.dumps(instance["num_gpu"])
elif PERFORMANCE_PRESET == "balanced":
    vllm_env["SM_VLLM_TENSOR_PARALLEL_SIZE"] = json.dumps(instance["num_gpu"])
    vllm_env["SM_VLLM_ENABLE_EXPERT_PARALLEL"] = "true"
elif PERFORMANCE_PRESET == "throughput":
    vllm_env["SM_VLLM_ENABLE_EXPERT_PARALLEL"] = "true"
    vllm_env["SM_VLLM_DATA_PARALLEL_SIZE"] = "4"
else:
    raise ValueError(f"unknown preset: {PERFORMANCE_PRESET}")

env = common_env | vllm_env
print(json.dumps(env, indent=2, ensure_ascii=False))

## 4. 모델 생성 → 엔드포인트 배포

In [ ]:
sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={"Image": inference_image, "Environment": env},
)

sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[{
        "VariantName": variant_name,
        "ModelName": model_name,
        "InstanceType": instance["type"],
        "InitialInstanceCount": 1,
        "ContainerStartupHealthCheckTimeoutInSeconds": 1800,  # 가중치 155 GiB 로딩 시간 감안
    }],
)

sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)
print(f"deploying endpoint: {endpoint_name}")

In [ ]:
# 가중치 로딩 + 워밍업 포함 약 10~15분 소요 (로컬 검증에서는 5~6분이었으나 인스턴스 콜드 스타트가 추가됨)
wait_for_endpoint(endpoint_name)
print("endpoint is InService")

## 5. 추론 테스트

### 5-1. 기본 텍스트 추론

In [ ]:
payload = {"messages": [{"role": "user", "content": "17 * 19 는 얼마야? 최종 정수만 답해."}]}

res = sm_runtime.invoke_endpoint(
    EndpointName=endpoint_name, Body=json.dumps(payload), ContentType="application/json"
)
response = json.loads(res["Body"].read().decode("utf-8"))
print(response["choices"][0]["message"]["content"])

### 5-2. Reasoning 모드 (Think High)

In [ ]:
payload = {
    "messages": [{"role": "user", "content": "17 * 19 를 계산 과정과 함께 설명해줘."}],
    "chat_template_kwargs": {"thinking": True, "reasoning_effort": "high"},
}

res = sm_runtime.invoke_endpoint(
    EndpointName=endpoint_name, Body=json.dumps(payload), ContentType="application/json"
)
response = json.loads(res["Body"].read().decode("utf-8"))
message = response["choices"][0]["message"]
print("reasoning:", message.get("reasoning"))
print("content:", message.get("content"))

### 5-3. 스트리밍

In [ ]:
class LineIterator:
    """SageMaker 스트리밍 응답의 바이트 청크를 줄 단위 SSE 이벤트로 재구성."""

    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = bytearray()

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            newline_pos = self.buffer.find(b"\n")
            if newline_pos >= 0:
                line = self.buffer[:newline_pos]
                self.buffer = self.buffer[newline_pos + 1:]
                if line:
                    return line
                continue
            chunk = next(self.byte_iterator)
            self.buffer.extend(chunk["PayloadPart"]["Bytes"])


payload = {
    "messages": [{"role": "user", "content": "짧은 시를 하나 지어줘."}],
    "stream": True,
}

res = sm_runtime.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name, Body=json.dumps(payload), ContentType="application/json"
)

for line in LineIterator(res["Body"]):
    text = line.decode("utf-8").removeprefix("data: ").strip()
    if text == "[DONE]" or not text:
        continue
    chunk = json.loads(text)
    delta = chunk["choices"][0].get("delta", {})
    if delta.get("content"):
        print(delta["content"], end="", flush=True)
print()

## Cleanup

> 🔴 **여기서 끝내지 마세요.** 실시간 엔드포인트는 이 셀을 실행해 삭제하기 전까지 계속 과금됩니다.

In [ ]:
sm.delete_endpoint(EndpointName=endpoint_name)
sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
sm.delete_model(ModelName=model_name)
print("deleted:", endpoint_name, endpoint_config_name, model_name)